# Day 2: Concepts, Labels, Dictionaries, and Supervised Classification

Today connects concepts to labels. We begin with authorship attribution as a classic supervised text problem, then compare rule-based measurement to supervised classifiers on SMS spam.

By the end you should be able to:

1. Explain authorship attribution as a text classification task.
2. Build a small dictionary or regex baseline.
3. Evaluate a classifier with precision, recall, F1, and a confusion matrix.
4. Compare dictionary, Naive Bayes, logistic regression, and linear SVM baselines.
5. Use stratified k-fold cross-validation and bias-variance diagnostics to assess generalization.
6. Inspect errors as part of measurement validation.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score, classification_report, make_scorer
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import LinearSVC

pd.set_option('display.max_colwidth', 140)

In [ ]:
from pathlib import Path


def find_data_dir():
    candidates = [Path('../data'), Path('materials/data'), Path('data')]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the course data directory.')


DATA_DIR = find_data_dir()
DATA_DIR

## spaCy setup

The notebooks use spaCy for tokenization and preprocessing. If `en_core_web_sm` is installed, the same setup also shows POS tags, lemmas, and dependency parses. If not, the notebooks still run with a blank English tokenizer.

In [ ]:
import spacy

try:
    nlp = spacy.load('en_core_web_sm')
    print('Loaded spaCy model: en_core_web_sm')
except OSError:
    nlp = spacy.blank('en')
    nlp.add_pipe('sentencizer')
    print('Using spacy.blank("en"). Install en_core_web_sm for POS tags, lemmas, and dependency parses.')


def spacy_preprocess(text, remove_stop=True, keep_alpha=True, min_len=2):
    """Tokenize and lightly preprocess text with spaCy."""
    doc = nlp.make_doc(str(text))
    tokens = []
    for token in doc:
        if keep_alpha and not token.is_alpha:
            continue
        if remove_stop and token.is_stop:
            continue
        term = token.text.lower()
        if len(term) >= min_len:
            tokens.append(term)
    return tokens


def spacy_analyzer(text):
    return spacy_preprocess(text)


def spacy_full_text_analyzer(text):
    return spacy_preprocess(text, remove_stop=False)


def spacy_analyzer_with_bigrams(text):
    tokens = spacy_preprocess(text)
    bigrams = [tokens[i] + '_' + tokens[i + 1] for i in range(len(tokens) - 1)]
    return tokens + bigrams


def spacy_token_table(text):
    """Show tokenization, preprocessing flags, and parses when a full model is available."""
    doc = nlp(str(text))
    rows = []
    for token in doc:
        rows.append({
            'text': token.text,
            'lemma': token.lemma_ or '(model needed)',
            'pos': token.pos_ or '(model needed)',
            'dep': token.dep_ or '(model needed)',
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'kept_by_preprocess': token.text.lower() in spacy_preprocess(token.text, remove_stop=True)
        })
    return pd.DataFrame(rows)

## 1. Federalist authorship attribution

We begin with the classic authorship example from the lecture. Train on essays known to be by Hamilton or Madison, then predict disputed essays.

The point is not just to get an answer. It is to see how the answer changes when we change the representation of the text. We start with a tiny stylometric representation based on the function words `man`, `by`, and `upon`, then compare it to a fuller bag-of-words representation.


In [ ]:
federalist = pd.read_csv(DATA_DIR / 'federalist.csv')

known = (
    federalist[federalist['author'].isin(['HAMILTON', 'MADISON'])]
    .sort_values('paper_id')
    .copy()
)
disputed = (
    federalist[federalist['author'].eq('HAMILTON OR MADISON')]
    .sort_values('paper_id')
    .copy()
)

known[['paper_id', 'title', 'author']].head()


### 1a. Function-word representation

Function words are useful in authorship attribution because they are frequent, hard to consciously control, and less directly tied to topic than content words. For each essay $i$ and function word $w$, we measure a rate per 1,000 tokens:

$$
x_{iw} = 1000 \times \frac{c_{iw}}{N_i}
$$

where $c_{iw}$ is the count of word $w$ in essay $i$ and $N_i$ is the total number of alphabetic tokens in the essay. Here we use only three historically salient words: `man`, `by`, and `upon`.


In [ ]:
function_words = ['man', 'by', 'upon']


def alpha_tokens(text):
    # Use spaCy's tokenizer so the feature construction matches the rest of the notebook.
    return [token.text.lower() for token in nlp.make_doc(str(text)) if token.is_alpha]


def function_word_rates(texts):
    rows = []
    for text in texts:
        tokens = alpha_tokens(text)
        total = max(len(tokens), 1)
        rows.append([1000 * tokens.count(word) / total for word in function_words])
    return np.asarray(rows, dtype=float)


def function_word_frame(frame):
    rates = pd.DataFrame(
        function_word_rates(frame['text']),
        columns=[f'{word}_per_1000' for word in function_words],
        index=frame.index,
    )
    return pd.concat([frame[['paper_id', 'title', 'author']], rates], axis=1)


known_function_words = function_word_frame(known)
disputed_function_words = function_word_frame(disputed)

known_function_words.groupby('author')[[f'{word}_per_1000' for word in function_words]].mean().round(2)


In [ ]:
known_long = known_function_words.melt(
    id_vars=['paper_id', 'author'],
    value_vars=[f'{word}_per_1000' for word in function_words],
    var_name='word',
    value_name='rate_per_1000',
)
known_long['word'] = known_long['word'].str.replace('_per_1000', '', regex=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=known_long, x='word', y='rate_per_1000', hue='author', errorbar='se', ax=axes[0])
axes[0].set_title('Function-word rates in known essays')
axes[0].set_xlabel('Function word')
axes[0].set_ylabel('Occurrences per 1,000 tokens')

sns.scatterplot(
    data=known_function_words,
    x='by_per_1000',
    y='upon_per_1000',
    hue='author',
    size='man_per_1000',
    sizes=(30, 180),
    alpha=0.8,
    ax=axes[1],
)
sns.scatterplot(
    data=disputed_function_words,
    x='by_per_1000',
    y='upon_per_1000',
    marker='X',
    color='black',
    s=90,
    label='disputed',
    ax=axes[1],
)
axes[1].set_title('Known and disputed essays in function-word space')
axes[1].set_xlabel('by per 1,000 tokens')
axes[1].set_ylabel('upon per 1,000 tokens')
axes[1].legend(loc='best')
plt.tight_layout()


We can now train a classifier using only these three rates. The model is deliberately small: it asks whether a transparent stylometric signal is enough to separate Hamilton from Madison.

$$
\Pr(y_i = \text{Madison} \mid x_i) = \operatorname{logit}^{-1}(\beta_0 + \beta_1 x_{i,man} + \beta_2 x_{i,by} + \beta_3 x_{i,upon})
$$


In [ ]:
function_word_model = Pipeline([
    ('features', FunctionTransformer(function_word_rates, validate=False)),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

function_word_model.fit(known['text'], known['author'])

function_word_disputed = disputed[['paper_id', 'title', 'author']].copy().reset_index(drop=True)
function_word_disputed['function_word_author'] = function_word_model.predict(disputed['text'])
function_word_disputed['function_word_confidence'] = function_word_model.predict_proba(disputed['text']).max(axis=1).round(3)
function_word_disputed


### 1b. Full-text representation

Now compare the function-word result with a much richer representation. Instead of three word rates, the model sees many spaCy-tokenized terms, including stop words, weighted by TF-IDF:

$$
\operatorname{tfidf}_{ij} = \operatorname{tf}_{ij} \times \log\left(\frac{N}{df_j}\right)
$$

This usually improves predictive flexibility, but it is also less transparent than the function-word model. Because this is authorship attribution, we keep stop words rather than removing them.


In [ ]:
full_text_author_model = Pipeline([
    ('vectorizer', TfidfVectorizer(analyzer=spacy_full_text_analyzer, min_df=2, max_features=3000)),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

full_text_author_model.fit(known['text'], known['author'])

federalist_comparison = disputed[['paper_id', 'title']].copy().reset_index(drop=True)
federalist_comparison['function_word_author'] = function_word_model.predict(disputed['text'])
federalist_comparison['function_word_confidence'] = function_word_model.predict_proba(disputed['text']).max(axis=1).round(3)
federalist_comparison['full_text_author'] = full_text_author_model.predict(disputed['text'])
federalist_comparison['full_text_confidence'] = full_text_author_model.predict_proba(disputed['text']).max(axis=1).round(3)
federalist_comparison['models_agree'] = (
    federalist_comparison['function_word_author'] == federalist_comparison['full_text_author']
)

federalist_comparison


In [ ]:
author_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
author_models = {
    'Function words only': function_word_model,
    'Full text TF-IDF': full_text_author_model,
}

author_cv_rows = []
for model_name, model in author_models.items():
    scores = cross_validate(
        model,
        known['text'],
        known['author'],
        cv=author_cv,
        scoring=['accuracy', 'f1_macro'],
    )
    for fold, (accuracy, f1_macro) in enumerate(
        zip(scores['test_accuracy'], scores['test_f1_macro']),
        start=1,
    ):
        author_cv_rows.append({
            'model': model_name,
            'fold': fold,
            'accuracy': accuracy,
            'macro_f1': f1_macro,
        })

author_cv_scores = pd.DataFrame(author_cv_rows)
author_cv_summary = (
    author_cv_scores
    .groupby('model')[['accuracy', 'macro_f1']]
    .agg(['mean', 'std'])
    .round(3)
)

author_cv_summary


In [ ]:
cv_plot = author_cv_scores.melt(
    id_vars=['model', 'fold'],
    value_vars=['accuracy', 'macro_f1'],
    var_name='metric',
    value_name='score',
)

plt.figure(figsize=(7, 4))
sns.pointplot(data=cv_plot, x='metric', y='score', hue='model', errorbar='sd', dodge=0.25)
plt.ylim(0, 1.03)
plt.title('Federalist authorship: representation comparison')
plt.xlabel('Cross-validation metric')
plt.ylabel('Held-out score')
plt.tight_layout()


Interpretation: if the two models agree on disputed essays, the conclusion is less dependent on one representation choice. If they disagree, that is a measurement warning: the answer may be driven by whether we prioritize stable style markers or broader topical vocabulary.


## 2. Dictionary measurement on TweetEval sentiment

Here we use a local sample from TweetEval sentiment. The concept is sentiment; the rule-based measurement is a small transparent lexicon. This gives us immediate labels for checking where the dictionary agrees or disagrees with human annotation.


In [ ]:
tweets = pd.read_csv(DATA_DIR / 'tweet_eval_sentiment_sample.csv')
tweets = tweets[['text', 'label_name', 'split']].dropna().copy()
tweets['label_name'] = tweets['label_name'].str.lower()

tweets[['text', 'label_name', 'split']].head()


## 2a. Inspect one tweet with spaCy

For dictionary work, spaCy is useful for seeing tokens and basic preprocessing decisions before writing search rules. Short social media text is deliberately messy: mentions, hashtags, punctuation, and spelling variation all affect what a dictionary can see.


In [ ]:
spacy_token_table(tweets.loc[0, 'text'])


In [ ]:
positive_words = {
    'good', 'great', 'best', 'love', 'loved', 'like', 'happy', 'awesome',
    'excellent', 'amazing', 'beautiful', 'perfect', 'win', 'winning',
    'thanks', 'thank', 'fun', 'enjoy', 'proud'
}

negative_words = {
    'bad', 'worst', 'hate', 'hated', 'sad', 'angry', 'awful', 'terrible',
    'horrible', 'poor', 'wrong', 'broken', 'kill', 'killing', 'dead',
    'disaster', 'problem', 'problems', 'fail', 'failed', 'fear'
}


def dictionary_sentiment(text):
    tokens = spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=2)
    positive_hits = sorted(set(token for token in tokens if token in positive_words))
    negative_hits = sorted(set(token for token in tokens if token in negative_words))
    score = len(positive_hits) - len(negative_hits)
    if score > 0:
        predicted = 'positive'
    elif score < 0:
        predicted = 'negative'
    else:
        predicted = 'neutral'
    return pd.Series({
        'dictionary_score': score,
        'dictionary_label': predicted,
        'positive_hits': positive_hits,
        'negative_hits': negative_hits,
    })

sentiment_rules = tweets['text'].apply(dictionary_sentiment)
tweets = pd.concat([tweets, sentiment_rules], axis=1)

tweets[['text', 'label_name', 'dictionary_label', 'dictionary_score', 'positive_hits', 'negative_hits']].head(12)


In [ ]:
print(tweets['label_name'].value_counts())
print('\nDictionary predictions:')
print(tweets['dictionary_label'].value_counts())

pd.crosstab(
    tweets['label_name'],
    tweets['dictionary_label'],
    normalize='index'
).round(3)


Discussion: where does the dictionary fail? Are the false positives caused by sarcasm, topic words, negation, slang, missing vocabulary, or ambiguous sentiment words?


## 2b. Dictionary scores by human label

A useful diagnostic is whether the rule-based score moves in the expected direction across human labels. If the score barely differs by label, the dictionary is probably not measuring the target concept well.


In [ ]:
sentiment_summary = (
    tweets
    .groupby('label_name')
    .agg(
        mean_dictionary_score=('dictionary_score', 'mean'),
        positive_rate=('dictionary_label', lambda x: (x == 'positive').mean()),
        negative_rate=('dictionary_label', lambda x: (x == 'negative').mean()),
        n=('dictionary_label', 'size')
    )
    .reindex(['negative', 'neutral', 'positive'])
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(data=sentiment_summary, x='label_name', y='mean_dictionary_score', color='#4C72B0', ax=axes[0])
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Mean dictionary score by human label')
axes[0].set_xlabel('Human label')
axes[0].set_ylabel('Positive hits minus negative hits')

sns.heatmap(
    pd.crosstab(tweets['label_name'], tweets['dictionary_label'], normalize='index').reindex(index=['negative', 'neutral', 'positive'], columns=['negative', 'neutral', 'positive']),
    annot=True,
    fmt='.2f',
    cmap='Blues',
    ax=axes[1]
)
axes[1].set_title('Dictionary label vs. human label')
axes[1].set_xlabel('Dictionary label')
axes[1].set_ylabel('Human label')

plt.tight_layout()
sentiment_summary.round(3)


In [ ]:
# Read a few disagreements. This is where measurement validation starts.
disagreements = tweets[tweets['label_name'] != tweets['dictionary_label']].copy()
disagreements[['text', 'label_name', 'dictionary_label', 'dictionary_score', 'positive_hits', 'negative_hits']].sample(10, random_state=4)


## 3. Labeled data: SMS spam

The SMS Spam Collection has human labels for short messages. It is a useful classroom dataset because it is local, imbalanced, and close to a real moderation/filtering task.

In [ ]:
sms = pd.read_csv(DATA_DIR / 'sms_spam.csv')
sms = sms[['text', 'label', 'is_spam']].dropna().copy()
sms['label'] = sms['label'].str.lower()

plt.figure(figsize=(5, 3.5))
sns.countplot(data=sms, x='label', order=['ham', 'spam'], color='#4C72B0')
plt.title('SMS label balance')
plt.xlabel('Human label')
plt.ylabel('Messages')
plt.tight_layout()

sms[['text', 'label']].head(), sms['label'].value_counts()

In [ ]:
spam_words = {
    'free', 'win', 'winner', 'won', 'prize', 'cash', 'urgent', 'claim',
    'txt', 'reply', 'call', 'mobile', 'ringtone', 'stop', 'guaranteed'
}


def dictionary_spam(text, threshold=1):
    tokens = spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=1)
    score = sum(token in spam_words for token in tokens)
    return 'spam' if score >= threshold else 'ham'

sms['dict_prediction'] = sms['text'].apply(dictionary_spam)
display(sms[['text', 'label', 'dict_prediction']].head())
pd.crosstab(sms['label'], sms['dict_prediction'], normalize='index').round(3)

## 4. A small evaluation helper

This is intentionally short. The goal is to make model comparison readable, not to hide the work in a package.

### Methodology formulas: labels, prediction, and evaluation

A dictionary classifier maps text to a score by counting words from a theory-driven set $D$:

$$s_i = \sum_{w \in x_i} \mathbb{1}(w \in D), \qquad \hat{y}_i = \mathbb{1}(s_i > \tau).$$

Multinomial Naive Bayes chooses the class with the largest posterior log score:

$$\hat{c} = \arg\max_c \left[\log P(c) + \sum_v x_{iv}\log P(v \mid c)\right].$$

Logistic regression estimates a conditional probability from text features:

$$P(y_i=1 \mid x_i) = \sigma(\beta_0 + x_i^\top\beta), \qquad \sigma(z)=\frac{1}{1+e^{-z}}.$$

A linear SVM learns a separating hyperplane and predicts by the sign of the margin:

$$\hat{y}_i = \operatorname{sign}(w^\top x_i + b).$$

The notebook evaluates predictions with precision, recall, and F1:

$$\mathrm{precision}=\frac{TP}{TP+FP}, \qquad \mathrm{recall}=\frac{TP}{TP+FN}, \qquad F_1=2\frac{\mathrm{precision}\times\mathrm{recall}}{\mathrm{precision}+\mathrm{recall}}.$$

For an advanced class, the key point is that model accuracy is not the same as measurement validity: the learned label $\hat{y}$ is only useful if the training labels $y$ actually capture the target concept.


In [ ]:
def evaluate_predictions(y_true, y_pred, positive_label='spam'):
    return pd.Series({
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        'recall': recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        'f1': f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)
    })

evaluate_predictions(sms['label'], sms['dict_prediction'])

In [ ]:
ConfusionMatrixDisplay.from_predictions(sms['label'], sms['dict_prediction'], cmap='Blues')
plt.title('Dictionary spam baseline')
plt.tight_layout()

## 5. Train and test split

Keep a held-out test set. We only look at test performance after training.

In [ ]:
train_df, test_df = train_test_split(
    sms,
    test_size=0.25,
    random_state=42,
    stratify=sms['label']
)

train_df.shape, test_df.shape

In [ ]:
nb_model = Pipeline([
    ('vectorizer', CountVectorizer(analyzer=spacy_analyzer, min_df=2, max_features=5000)),
    ('classifier', MultinomialNB())
])

logit_model = Pipeline([
    ('vectorizer', TfidfVectorizer(analyzer=spacy_analyzer_with_bigrams, min_df=2, max_features=7000)),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

svm_model = Pipeline([
    ('vectorizer', TfidfVectorizer(analyzer=spacy_analyzer_with_bigrams, min_df=2, max_features=7000)),
    ('classifier', LinearSVC(class_weight='balanced', random_state=42))
])

models = {
    'naive_bayes_counts': nb_model,
    'logistic_tfidf': logit_model,
    'linear_svm_tfidf': svm_model,
}

for model in models.values():
    model.fit(train_df['text'], train_df['label'])

test_df = test_df.copy()
test_df['nb_prediction'] = nb_model.predict(test_df['text'])
test_df['logit_prediction'] = logit_model.predict(test_df['text'])
test_df['svm_prediction'] = svm_model.predict(test_df['text'])
test_df['dict_prediction'] = test_df['text'].apply(dictionary_spam)


In [ ]:
results = pd.DataFrame({
    'dictionary': evaluate_predictions(test_df['label'], test_df['dict_prediction']),
    'naive_bayes_counts': evaluate_predictions(test_df['label'], test_df['nb_prediction']),
    'logistic_tfidf': evaluate_predictions(test_df['label'], test_df['logit_prediction']),
    'linear_svm_tfidf': evaluate_predictions(test_df['label'], test_df['svm_prediction']),
})
results.round(3)


## 5a. Stratified k-fold cross-validation

The lecture separates training, validation, and testing. Cross-validation repeats the validation step across multiple folds:

$$\widehat{M}_{CV} = \frac{1}{k}\sum_{j=1}^{k} M_j.$$

For imbalanced classification, we use stratified folds so that each fold has a similar spam/ham ratio. In applied work, use cross-validation on the training data for model selection, then reserve the test set for the final estimate.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, pos_label='spam', zero_division=0),
    'recall': make_scorer(recall_score, pos_label='spam', zero_division=0),
    'f1': make_scorer(f1_score, pos_label='spam', zero_division=0),
}

cv_rows = []
for name, model in models.items():
    scores = cross_validate(
        model,
        train_df['text'],
        train_df['label'],
        cv=cv,
        scoring=scoring,
        n_jobs=None,
        return_train_score=True,
    )
    for fold in range(cv.get_n_splits()):
        cv_rows.append({
            'model': name,
            'fold': fold + 1,
            'accuracy': scores['test_accuracy'][fold],
            'precision': scores['test_precision'][fold],
            'recall': scores['test_recall'][fold],
            'f1': scores['test_f1'][fold],
            'train_f1': scores['train_f1'][fold],
        })

cv_results = pd.DataFrame(cv_rows)
cv_summary = (
    cv_results
    .groupby('model')[['accuracy', 'precision', 'recall', 'f1', 'train_f1']]
    .agg(['mean', 'std'])
    .round(3)
)
cv_summary


The fold-to-fold spread is often as informative as the mean. Large variation suggests that performance depends strongly on which examples happen to be in the training set.


In [ ]:
plt.figure(figsize=(7, 4))
sns.lineplot(data=cv_results, x='fold', y='f1', hue='model', marker='o')
plt.ylim(0, 1)
plt.title('Stratified 5-fold cross-validation: spam F1')
plt.xlabel('Fold')
plt.ylabel('Validation F1')
plt.tight_layout()


## 5b. Bias-variance diagnostics

The bias-variance tradeoff is a generalization story. For squared error, the standard decomposition is:

$$\mathbb{E}\left[(Y-\hat{f}(X))^2\right] = \mathrm{Bias}^2 + \mathrm{Variance} + \mathrm{Irreducible\ error}.$$

For classification, we use the idea diagnostically rather than literally: low train and validation performance suggests underfitting; high train performance but lower validation performance suggests overfitting.


In [ ]:
bv_rows = []
for name, model in models.items():
    train_pred = model.predict(train_df['text'])
    heldout_pred = model.predict(test_df['text'])
    bv_rows.extend([
        {
            'model': name,
            'split': 'train',
            'accuracy': accuracy_score(train_df['label'], train_pred),
            'f1': f1_score(train_df['label'], train_pred, pos_label='spam', zero_division=0),
        },
        {
            'model': name,
            'split': 'held-out test',
            'accuracy': accuracy_score(test_df['label'], heldout_pred),
            'f1': f1_score(test_df['label'], heldout_pred, pos_label='spam', zero_division=0),
        },
    ])

bv_diagnostics = pd.DataFrame(bv_rows)
bv_gap = (
    bv_diagnostics
    .pivot(index='model', columns='split', values='f1')
    .assign(generalization_gap=lambda d: d['train'] - d['held-out test'])
    .round(3)
)
bv_gap


In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=bv_diagnostics, x='model', y='f1', hue='split')
plt.ylim(0, 1)
plt.title('Bias-variance diagnostic: train vs held-out F1')
plt.xlabel('Model')
plt.ylabel('Spam F1')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()


## 5c. Model comparison visuals

Tables are precise, but plots make tradeoffs visible. Here we compare held-out metrics and precision-recall behavior across models. The linear SVM produces margin scores rather than probabilities, but precision-recall curves can use any score that ranks likely positives above likely negatives.


In [ ]:
plt.figure(figsize=(8, 3.5))
sns.heatmap(results, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1)
plt.title('Held-out classification metrics')
plt.tight_layout()

In [ ]:
def positive_class_scores(model, texts, positive_label='spam'):
    # Return probability or margin scores oriented toward the positive class.
    classes = list(model.classes_)
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(texts)[:, classes.index(positive_label)]
    scores = model.decision_function(texts)
    if getattr(scores, 'ndim', 1) > 1:
        return scores[:, classes.index(positive_label)]
    return scores if classes[-1] == positive_label else -scores


y_true_binary = (test_df['label'] == 'spam').astype(int)
model_scores = {
    'Naive Bayes': positive_class_scores(nb_model, test_df['text']),
    'Logistic TF-IDF': positive_class_scores(logit_model, test_df['text']),
    'Linear SVM TF-IDF': positive_class_scores(svm_model, test_df['text']),
}

plt.figure(figsize=(6, 5))
for name, scores in model_scores.items():
    precision, recall, _ = precision_recall_curve(y_true_binary, scores)
    ap = average_precision_score(y_true_binary, scores)
    plt.plot(recall, precision, label=f'{name} (AP={ap:.3f})')

baseline = y_true_binary.mean()
plt.axhline(baseline, color='gray', linestyle='--', label=f'Baseline spam rate={baseline:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-recall curves for spam detection')
plt.legend()
plt.tight_layout()


### Additional demo: calibration and threshold choice

The same predicted probabilities can support different decisions. Here the threshold is the probability at which a message is sent to the spam folder, so false positives and false negatives have different substantive costs.

In [ ]:
from sklearn.calibration import calibration_curve

logit_scores = model_scores['Logistic TF-IDF']
prob_true, prob_pred = calibration_curve(y_true_binary, logit_scores, n_bins=8, strategy='quantile')

threshold_rows = []
for threshold in np.linspace(0.20, 0.80, 7):
    pred = (logit_scores >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_true_binary, pred, zero_division=0),
        'recall': recall_score(y_true_binary, pred, zero_division=0),
        'f1': f1_score(y_true_binary, pred, zero_division=0),
        'positive_rate': pred.mean()
    })
threshold_table = pd.DataFrame(threshold_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
axes[0].plot(prob_pred, prob_true, marker='o', label='Logistic TF-IDF')
axes[0].set_title('Calibration curve')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Observed positive rate')
axes[0].legend()

threshold_long = threshold_table.melt(
    id_vars='threshold',
    value_vars=['precision', 'recall', 'f1', 'positive_rate'],
    var_name='metric',
    value_name='value'
)
sns.lineplot(data=threshold_long, x='threshold', y='value', hue='metric', marker='o', ax=axes[1])
axes[1].set_title('Changing the classification threshold')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Metric value')

plt.tight_layout()
threshold_table.round(3)

In [ ]:
ConfusionMatrixDisplay.from_predictions(test_df['label'], test_df['logit_prediction'], cmap='Blues')
plt.title('Logistic regression with TF-IDF')
plt.tight_layout()

## 6. Error analysis

Read errors before trusting a model. Errors are often substantively informative.

In [ ]:
def show_errors(
    df,
    prediction_col,
    true_col='label',
    text_col='text',
    n=8,
    kind='false_positive',
    positive_label='spam',
    negative_label='ham'
):
    if kind == 'false_positive':
        subset = df[(df[true_col] == negative_label) & (df[prediction_col] == positive_label)]
    elif kind == 'false_negative':
        subset = df[(df[true_col] == positive_label) & (df[prediction_col] == negative_label)]
    else:
        subset = df[df[true_col] != df[prediction_col]]
    return subset[[text_col, true_col, prediction_col]].sample(min(n, len(subset)), random_state=2)

show_errors(test_df, 'logit_prediction', kind='false_positive')

In [ ]:
show_errors(test_df, 'logit_prediction', kind='false_negative')

## 7. What did the classifier learn?

For this binary logistic classifier, positive coefficients point toward spam and negative coefficients point toward ham.

In [ ]:
feature_names = logit_model.named_steps['vectorizer'].get_feature_names_out()
coefs = logit_model.named_steps['classifier'].coef_[0]

features = pd.DataFrame({'term': feature_names, 'coefficient': coefs})
pd.concat([
    features.sort_values('coefficient', ascending=False).head(15),
    features.sort_values('coefficient', ascending=True).head(15)
])

## 7a. Coefficient plot

Feature weights are not causal effects, but they help diagnose what a classifier is using to separate classes.

In [ ]:
coef_plot = pd.concat([
    features.sort_values('coefficient', ascending=True).head(15),
    features.sort_values('coefficient', ascending=False).head(15)
]).sort_values('coefficient')

plt.figure(figsize=(8, 8))
colors = np.where(coef_plot['coefficient'] > 0, '#4C72B0', '#C44E52')
plt.barh(coef_plot['term'], coef_plot['coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title('Most influential TF-IDF terms in spam classifier')
plt.xlabel('Coefficient toward spam')
plt.tight_layout()

## 8. Annotation budget learning curve

A learning curve connects model performance to the amount of labeled data. This is a practical way to discuss whether more annotation is likely to help.

In [ ]:
learning_seed = 42
candidate_sizes = [50, 100, 250, 500, 1000, min(2000, len(train_df))]
train_sizes = sorted({size for size in candidate_sizes if size <= len(train_df)})


def balanced_sample(frame, label_col, size, seed):
    per_class = max(1, size // frame[label_col].nunique())
    pieces = []
    for _, group in frame.groupby(label_col):
        pieces.append(group.sample(min(per_class, len(group)), random_state=seed))
    return pd.concat(pieces).sample(frac=1, random_state=seed)

learning_rows = []
for size in train_sizes:
    subset = balanced_sample(train_df, 'label', size, learning_seed + size)
    model = Pipeline([
        ('vectorizer', TfidfVectorizer(analyzer=spacy_analyzer_with_bigrams, min_df=2, max_features=3000)),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])
    model.fit(subset['text'], subset['label'])
    pred = model.predict(test_df['text'])
    learning_rows.append({
        'labeled_examples': len(subset),
        'accuracy': accuracy_score(test_df['label'], pred),
        'f1': f1_score(test_df['label'], pred, pos_label='spam', zero_division=0)
    })

learning_curve = pd.DataFrame(learning_rows)

plt.figure(figsize=(6, 4))
learning_long = learning_curve.melt(id_vars='labeled_examples', var_name='metric', value_name='value')
sns.lineplot(data=learning_long, x='labeled_examples', y='value', hue='metric', marker='o')
plt.ylim(0, 1)
plt.title('Learning curve for SMS spam data')
plt.xlabel('Number of labeled training examples')
plt.ylabel('Held-out performance')
plt.tight_layout()

learning_curve.round(3)

## 9. Student task

Choose one concept and one model. Then write a short validation note:

1. What is the concept?
2. What is the label or proxy?
3. Which baseline did you compare against?
4. Did cross-validation suggest stable performance?
5. Do train vs. held-out scores suggest underfitting or overfitting?
6. What errors matter most?
7. What would you inspect before using the model in a paper?


In [ ]:
# Your turn: edit the dictionary words or vectorizer settings, then rerun evaluation.
custom_spam_words = spam_words | {'voucher', 'bonus', 'selected'}

# Keep this cell as a workspace for your own model comparison.